# Quantum Grover Cost Estimator for Exponential Congruences
This notebook implements a classical **Meet-in-the-Middle** algorithm to solve exponential congruences and provides a theoretical framework to estimate the acceleration bounds when using **Grover's Quantum Search Algorithm**.

## 🧠 Mathematical & Quantum Framework

We are evaluating equations in the form:
$$a \cdot f^x + b \cdot g^y \equiv c \pmod{q}$$

### Complexity Comparison:
1. **Classical Space Optimization:** Instead of a brute-force $O(N)$ check where $N = (\text{max\_exp} + 1)^2$, this solver leverages a hash table lookup reducing time complexity asymptotically.
2. **Quantum Speedup Calculation:** If there are $M$ valid solutions, Grover's algorithm provides a quadratic speedup. The theoretical oracle calls required are calculated as:
$$\text{Grover Cost} \approx \frac{\pi}{4} \sqrt{\frac{N}{M}}$$

In [5]:
import math
from collections import defaultdict
from tabulate import tabulate

def solve_exp_congruence(a, b, c, f, g, q, max_exp=None):
    if max_exp is None:
        max_exp = q - 2

    table = defaultdict(list)
    val = 1
    for x in range(max_exp + 1):
        table[(a * val) % q].append(x)
        val = (val * f) % q

    solutions = []
    val = 1
    for y in range(max_exp + 1):
        rhs = (c - (b * val) % q) % q
        if rhs in table:
            for x in table[rhs]:
                solutions.append((x, y))
        val = (val * g) % q

    solutions = sorted(set(solutions))

    N = (max_exp + 1) ** 2
    M = len(solutions)
    classical_exp = (N + 1) / (M + 1)
    grover_exp    = (math.pi / 4) * math.sqrt(N / M) if M > 0 else float("inf")
    return solutions, N, M, classical_exp, grover_exp, max_exp

def format_num(x):
    if x == float("inf"):
        return "—"
    if isinstance(x, float):
        return f"{x:,.2f}"
    return f"{x:,.0f}"

def print_case(title, a, b, c, f, g, q, max_exp=None):
    solutions, N, M, E_class, E_grov, max_used = solve_exp_congruence(a, b, c, f, g, q, max_exp=max_exp)

    print("\n" + "=" * 70)
    print(f" 📌 PROJECT CASE: {title.upper()}")
    print("=" * 70)

    metadata = [
        ["Target Equation", f"{a}·{f}^x + {b}·{g}^y ≡ {c} (mod {q})"],
        ["Search Bound", f"x, y ∈ [0, {max_used}]"],
        ["Total Search Space (N)", f"{N:,} combinations"],
        ["Total Solutions Found (M)", f"{M:,}"]
    ]
    print(tabulate(metadata, headers=["Parameter", "Value"], tablefmt="fancy_grid"))
    print("\n")

    if M > 0:
        print("💡 VALID SOLUTIONS FOUND:")
        sol_table = [[f"#{i}", f"x = {x}", f"y = {y}"] for i, (x, y) in enumerate(solutions, 1)]
        print(tabulate(sol_table, headers=["Index", "X Value", "Y Value"], tablefmt="presto"))
    else:
        print("❌ STATUS: No valid solutions exist within the designated search space.")
    print("\n")

    cost_data = [
        ["⚡ Classical Lookup (Average)", f"{format_num(E_class)} evaluations", "O(√N) space/time"],
        ["🔮 Grover's Search (Theoretical)", f"{format_num(E_grov)} oracle calls", "O(√N/M) upper bound"]
    ]
    print("📊 ALGORITHMIC COST COMPARISON:")
    print(tabulate(cost_data, headers=["Approach / Framework", "Estimated Complexity Cost", "Asymptotic Bound"], tablefmt="fancy_grid"))
    print("=" * 70 + "\n")

## 📊 Running Benchmark Test Cases

Below, we run the solver against two distinct setups to verify theoretical bounds:
1. **Example 1:** Evaluates a high density of valid solutions to observe the Grover acceleration behavior.
2. **No-Solution Example:** Evaluates the boundary threshold when zero solutions exist to examine theoretical infinity upper limits.

In [6]:
# Execution of benchmark cases
print_case("Example 1 → 2^x + 3^y ≡ 10 (mod 17)", a=1, b=1, c=10, f=2, g=3, q=17)
print_case("No-solution → 4^x + 4^y ≡ 1 (mod 5)",   a=1, b=1, c=1,  f=4, g=4, q=5)


 📌 PROJECT CASE: EXAMPLE 1 → 2^X + 3^Y ≡ 10 (MOD 17)
╒═══════════════════════════╤═════════════════════════════╕
│ Parameter                 │ Value                       │
╞═══════════════════════════╪═════════════════════════════╡
│ Target Equation           │ 1·2^x + 1·3^y ≡ 10 (mod 17) │
├───────────────────────────┼─────────────────────────────┤
│ Search Bound              │ x, y ∈ [0, 15]              │
├───────────────────────────┼─────────────────────────────┤
│ Total Search Space (N)    │ 256 combinations            │
├───────────────────────────┼─────────────────────────────┤
│ Total Solutions Found (M) │ 16                          │
╘═══════════════════════════╧═════════════════════════════╛


💡 VALID SOLUTIONS FOUND:
 Index   | X Value   | Y Value
---------+-----------+-----------
 #1      | x = 0     | y = 2
 #2      | x = 1     | y = 10
 #3      | x = 2     | y = 15
 #4      | x = 3     | y = 14
 #5      | x = 4     | y = 7
 #6      | x = 5     | y = 13
 #7      | x = 6